In [1]:
import pandas as pd
df = pd.read_excel('./data/apt_seoul.xlsx',skiprows=12,thousands=",")

C:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#전처리 과정
1. 아파트데이터 분할(시,군,구)
2. 거래금액 단위 변환(억단위)
3. 번지, 본번, 부번 제거
4. 전용면적 제곱미터 -> 평으로 변환
5. 해제사유발생일이 정상치인 행을 삭제(거래 취소 확정 -> 필요X)
6. 계약년월로 요일을 구하여 요일별 가격(요일칼럼 추가)

In [2]:
df

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
0,1,서울특별시 은평구 수색동,416-1,416,1,DMC 진흥,78.2853,202604,4,77500,...,15,개인,개인,2003,은평터널로 60,-,중개거래,서울 은평구,-,아파트
1,2,서울특별시 서대문구 북가좌동,481,481,0,DMC래미안e편한세상,153.8600,202604,3,165000,...,8,개인,개인,2012,수색로 100,-,직거래,-,-,아파트
2,3,서울특별시 서대문구 대현동,144,144,0,신촌럭키,59.7000,202604,3,120000,...,13,개인,개인,1999,이화여대길 50-12,-,중개거래,서울 서대문구,-,아파트
3,4,서울특별시 성동구 성수동1가,708,708,0,동아그린,58.3200,202604,3,135000,...,12,개인,개인,1999,상원길 76,-,중개거래,서울 성동구,-,아파트
4,5,서울특별시 송파구 잠실동,22,22,0,리센츠,84.9900,202604,3,350000,...,11,개인,개인,2008,올림픽로 135,-,중개거래,서울 송파구,-,아파트
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76661,76662,서울특별시 성북구 하월곡동,222,222,0,월곡두산위브,84.8430,202504,7,81800,...,9,개인,개인,2003,오패산로 46,-,중개거래,서울 성북구,25.06.16,아파트
76662,76663,서울특별시 동대문구 장안동,580,580,0,장안위더스빌,84.8500,202504,7,77800,...,16,개인,개인,2011,한천로 42,-,중개거래,서울 동대문구,25.07.22,아파트
76663,76664,서울특별시 성북구 안암동1가,361,361,0,안암동삼성래미안,59.6983,202504,7,76000,...,6,개인,개인,2005,고려대로17가길 64,-,중개거래,서울 성북구,25.06.16,아파트
76664,76665,서울특별시 성북구 정릉동,780,780,0,"산장아파트가동,나동",52.8900,202504,7,30000,...,1,개인,개인,1977,보국문로 167,-,중개거래,서울 성북구,25.06.30,아파트


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76666 entries, 0 to 76665
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   NO        76666 non-null  int64  
 1   시군구       76666 non-null  object 
 2   번지        76666 non-null  object 
 3   본번        76666 non-null  int64  
 4   부번        76666 non-null  int64  
 5   단지명       76666 non-null  object 
 6   전용면적(㎡)   76666 non-null  float64
 7   계약년월      76666 non-null  int64  
 8   계약일       76666 non-null  int64  
 9   거래금액(만원)  76666 non-null  int64  
 10  동         76666 non-null  object 
 11  층         76666 non-null  int64  
 12  매수자       76666 non-null  object 
 13  매도자       76666 non-null  object 
 14  건축년도      76666 non-null  int64  
 15  도로명       76666 non-null  object 
 16  해제사유발생일   76666 non-null  object 
 17  거래유형      76666 non-null  object 
 18  중개사소재지    76666 non-null  object 
 19  등기일자      76666 non-null  object 
 20  주택유형      76666 non-null  ob

In [4]:
df.loc[[ 3 ]]

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
3,4,서울특별시 성동구 성수동1가,708,708,0,동아그린,58.32,202604,3,135000,...,12,개인,개인,1999,상원길 76,-,중개거래,서울 성동구,-,아파트


In [5]:
sum(df['단지명'].str.contains('선사현대'))
r1 = df['단지명'].str.contains('선사현대')
df.loc[ r1 ]

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
936,937,서울특별시 강동구 암사동,509,509,0,선사현대아파트,59.64,202603,21,137500,...,8,개인,개인,2000,상암로 11,-,중개거래,"서울 강동구, 서울 송파구",-,아파트
3058,3059,서울특별시 강동구 암사동,509,509,0,선사현대아파트,59.64,202603,6,143900,...,28,개인,개인,2000,상암로 11,-,중개거래,서울 강동구,26.03.20,아파트
4180,4181,서울특별시 강동구 암사동,509,509,0,선사현대아파트,82.94,202602,27,200000,...,15,개인,개인,2000,상암로 11,-,중개거래,서울 강동구,-,아파트
4643,4644,서울특별시 강동구 암사동,509,509,0,선사현대아파트,59.64,202602,25,144000,...,3,개인,개인,2000,상암로 11,-,중개거래,서울 강동구,-,아파트
5340,5341,서울특별시 강동구 암사동,509,509,0,선사현대아파트,59.64,202602,21,144500,...,2,개인,개인,2000,상암로 11,-,중개거래,"서울 강동구, 서울 송파구",-,아파트
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76129,76130,서울특별시 강동구 암사동,509,509,0,선사현대아파트,83.67,202504,10,127000,...,14,개인,개인,2000,상암로 11,-,중개거래,서울 강동구,25.05.16,아파트
76130,76131,서울특별시 강동구 암사동,509,509,0,선사현대아파트,83.79,202504,10,122000,...,16,개인,개인,2000,상암로 11,-,중개거래,서울 강동구,25.07.10,아파트
76315,76316,서울특별시 강동구 암사동,509,509,0,선사현대아파트,59.64,202504,9,104000,...,10,개인,개인,2000,상암로 11,-,중개거래,서울 강동구,25.06.27,아파트
76316,76317,서울특별시 강동구 암사동,509,509,0,선사현대아파트,72.85,202504,9,113000,...,21,개인,개인,2000,상암로 11,-,중개거래,서울 강동구,25.06.20,아파트


In [6]:
df.describe()

,NO,본번,부번,전용면적(㎡),계약년월,계약일,거래금액(만원),층,건축년도
count,76666.000000,76666.000000,76666.000000,76666.000000,76666.000000,76666.000000,7.666600e+04,76666.000000,76666.00000
mean,38333.500000,639.730924,4.548327,75.265764,202526.253359,16.450187,1.201652e+05,9.585670,2003.25032
std,22131.712206,632.316776,39.419937,27.371606,37.503080,8.523682,8.870981e+04,6.375242,11.33792
min,1.000000,1.000000,0.000000,11.330000,202504.000000,1.000000,6.500000e+03,-2.000000,1961.00000
25%,19167.250000,257.250000,0.000000,59.750100,202506.000000,10.000000,6.800000e+04,5.000000,1996.00000
50%,38333.500000,510.000000,0.000000,79.980000,202509.000000,17.000000,9.850000e+04,9.000000,2003.00000
75%,57499.750000,822.000000,0.000000,84.960000,202512.000000,24.000000,1.470000e+05,13.000000,2012.00000
max,76666.000000,4974.000000,2837.000000,317.360000,202604.000000,31.000000,2.900000e+06,66.000000,2025.00000


In [14]:
# 계약년월별 계약건수 - count(). 건수는 행갯수라서 아무 칼럼 선택
df.groupby('계약년월')['층'].count().sort_values() #오름차순
df.groupby('계약년월')['층'].count().sort_values(ascending=False) #내림차순

계약년월
202506    12611
202509     9080
202510     8941
202505     8560
202602     5851
202601     5496
202512     4916
202504     4704
202507     4635
202508     4612
202603     3676
202511     3519
202604       65
Name: 층, dtype: int64

In [18]:
#층별 건수
df.groupby('층')['층'].count().sort_values()

층
-2        1
 63       1
 61       1
 60       1
 59       1
       ... 
 6     4775
 7     4822
 4     4912
 3     4938
 5     5117
Name: 층, Length: 65, dtype: int64

In [20]:
df.['전용면적(㎡)'].plot(kind='box')

SyntaxError: invalid syntax (2563463609.py, line 1)